In [5]:
from openai import OpenAI
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

load_dotenv()

client = OpenAI()
llm = ChatOpenAI(model = "gpt-5-nano", temperature = 0)

In [6]:
# ── RAG Prompt Pattern ─────────────────────────────────────────────────────
rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a helpful assistant.
         Answer using ONLY the context below.
         If the answer isn't in the context, say "I don't have that information."
         Never use outside knowledge.
         Context:
         {context}"""),
        ("human", "{question}"),
    ]
)

In [7]:
# ── Format retrieved context ─────────────────────────────────────────────────────
def format_context(docs: list[dict]) -> str:
    if not docs:
        return "No context retrieved."
    return "\n\n---\n\n".join([
        f"[Source: {i+1}]: {doc['text']}" for i, doc in enumerate(docs)
    ])

In [8]:
# ── Full RAG chain ─────────────────────────────────────────────────────
from regex import R


def mock_retriver(question: str) -> list[dict]:
    return [
        {"text": "PostgreSQL B-tree indexes support equality and range queries."},
        {"text": "Use CREATE INDEX CONCURRENTLY to avoid table locks."},
    ]

rag_chain = (
    {
        "context": RunnableLambda(lambda x: format_context(mock_retriver(x["question"]))),
        "question": RunnablePassthrough() | RunnableLambda(lambda x: x["question"]),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)



In [9]:
answer = rag_chain.invoke({"question": "How do I add an index without locking the table?"})
print(f"Answer : {answer}")

Answer : Use CREATE INDEX CONCURRENTLY to avoid table locks.
